# Day 03: Çok Kaynaklı Veri İşleme Hattı (Pandas, CSV/JSON Parser & Normalizer)

**Merinos Industrial AI Internship Portfolio — Day 03**  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Lisans:** Özel Lisans — Tüm Hakları Saklıdır (All Rights Reserved)  

---

## 1. Problem Tanımı ve Mühendislik Motivasyonu

Endüstriyel bir tekstil kampüsünde veriler tek bir temiz kaynaktan gelmez. Merinos tesislerinde:
- **Tezgah Hattı (PLC/SCADA):** Boyutları santimetre veya inç olarak loglayan, ham CSV üreten sensör sistemleri,
- **Tasarım & Ar-Ge:** Desen adları, renk paletleri (Hex) ve koleksiyon bilgilerini içeren JSON katalog beslemeleri,
- **ERP / Depo Yönetimi:** İplik hammadde kompozisyonu ve stok durumunu tutan ilişkisel tablolar mevcuttur.

Bu kaynaklar birleştirilirken eksik birincil anahtarlar, farklı metrik/emperyal birimler (inç vs. cm), tutarsız metinler (büyük/küçük harf, gereksiz boşluklar) ve geçersiz renk kodları boru hattını bozar.

Bu çalışmada, **Pandas** tabanlı hata toleranslı bir ayrıştırma (parsing), birleştirme (merge), normalizasyon ve karantina mimarisi kuruyoruz.

## 2. Neden Önemli? (Endüstriyel Etki & İş Değeri)

- **Veri Güvenilirliği:** Hatalı birim dönüşümü (örneğin 100 inç halının 100 cm olarak sisteme kaydedilmesi) üretim planlamasında %154 geometrik hata yaratır.
- **Karantina Ayrımı (Data Quarantine):** Bozuk bir satır tüm boru hattını çökertmemeli; geçerli kayıtlar işlenmeye devam ederken kirli veriler izole edilip loglanmalıdır.
- **Aşağı Yönlü (Downstream) Modellere Hazırlık:** Day 02 Pydantic modellerine uygun, temiz ve doğrulanmış veri seti sağlanır.

## 3. Matematiksel & İstatistiksel Temeller

1. **Birim Dönüşümü (Unit Conversion):**
$$D_{\text{cm}} = D_{\text{inch}} \times 2.54$$
$$D_{\text{cm}} = D_{\text{mm}} \times 0.1$$

2. **Veri Tamlığı ve Başarı Oranı (Data Quality Score, $Q$):**
$$Q = \left( \frac{N_{\text{valid}}}{N_{\text{merged}}} \right) \times 100$$
burada $N_{\text{valid}}$ tüm kısıtlardan geçen geçerli kayıt adedi, $N_{\text{merged}}$ birleştirilmiş tekil ürün adedidir.

3. **Kayıp Değer Analizi (Missingness Rate, $\mathcal{M}$):**
$$\mathcal{M}_j = \frac{1}{N} \sum_{i=1}^N \mathbb{I}(x_{ij} = \text{null})$$

## 4. Kütüphane İncelemesi: Pandas vs Polars vs DuckDB

| Kriter | Pandas 2.x | Polars | DuckDB |
|---|---|---|---|
| Çalışma Modeli | Eager (Bellek İçi) | Lazy & Eager | SQL / Out-of-Core |
| Çoklu Çekirdek (Multi-threading) | Sınırlı (GIL) | Mükemmel (Rust Rayon) | Mükemmel (C++ Vektörize) |
| Ekosistem Olgunluğu | Çok Yüksek (Python standardı) | Yüksek / Hızla artıyor | Yüksek (Analitik SQL) |
| Pydantic v2 Entegrasyonu | Doğrudan dict dökümü | `to_dicts()` ile | Python nesnesi dönüşümü gerekir |
| Bu Günde Seçim Gerekçesi | Endüstriyel yaygınlık ve Pydantic v2 ile birebir uyum için **Pandas** tercih edilmiştir. |

In [1]:
import sys
from pathlib import Path

# Dinamik kök dizin ekleme (day03 modülleri için)
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.startswith('day') else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 5. Minimal Çalışır Kod: Veri Kaynaklarının Yüklenmesi ve Birleştirilmesi
import io
import json
import pandas as pd

# Örnek üretim logu CSV (Bellek içi simülasyon)
csv_data = """product_id,raw_width,raw_length,dimension_unit,pile_height_mm
MRP-1001,160.0,230.0,cm,11.5
MRP-1002,78.74,118.11,inch,12.0
MRP-1003,-50.0,200.0,cm,10.0
"""

# Örnek katalog JSON beslemesi
json_data = [
    {"product_code": "MRP-1001", "product_name": "Prestij Venedik", "composition": "wool", "palette": ["#2B3A42", "#4F6D7A"]},
    {"product_code": "MRP-1002", "product_name": "Uşak Madalyon", "composition": "acrylic", "palette": ["#FFFFFF", "#000000"]},
    {"product_code": "MRP-1003", "product_name": "Hatalı Ürün", "composition": "wool", "palette": ["#INVALID"]}
]

df_csv = pd.read_csv(io.StringIO(csv_data))
df_json = pd.DataFrame(json_data)
df_json.rename(columns={"product_code": "product_id"}, inplace=True)

merged = pd.merge(df_csv, df_json, on="product_id", how="outer")
print("Birleştirilmiş Ham Veri Seti:")
print(merged)

Birleştirilmiş Ham Veri Seti:
  product_id  raw_width  raw_length dimension_unit  pile_height_mm  \
0   MRP-1001     160.00      230.00             cm            11.5   
1   MRP-1002      78.74      118.11           inch            12.0   
2   MRP-1003     -50.00      200.00             cm            10.0   

      product_name composition             palette  
0  Prestij Venedik        wool  [#2B3A42, #4F6D7A]  
1    Uşak Madalyon     acrylic  [#FFFFFF, #000000]  
2      Hatalı Ürün        wool          [#INVALID]  


In [2]:
# Normalizasyon ve Birim Dönüşüm Fonksiyonları
def convert_to_cm(val, unit):
    if unit == "inch":
        return round(val * 2.54, 2)
    return round(val, 2)

merged["width_cm"] = merged.apply(lambda r: convert_to_cm(r["raw_width"], r["dimension_unit"]), axis=1)
merged["length_cm"] = merged.apply(lambda r: convert_to_cm(r["raw_length"], r["dimension_unit"]), axis=1)

print("Birimleri Normalize Edilmiş Veri:")
print(merged[["product_id", "width_cm", "length_cm", "pile_height_mm"]])

Birimleri Normalize Edilmiş Veri:
  product_id  width_cm  length_cm  pile_height_mm
0   MRP-1001     160.0      230.0            11.5
1   MRP-1002     200.0      300.0            12.0
2   MRP-1003     -50.0      200.0            10.0


## 6. Deneyler & Parametre Analizi

Büyük ölçekli üretim loglarını parse ederken Pandas `read_csv` ile chunking (parça parça okuma) yönteminin bellek tasarrufunu simüle edelim.

In [3]:
import timeit

# Sentetik büyük veri simülasyonu (10,000 satır)
large_data = "product_id,raw_width,raw_length,dimension_unit\n" + "\n".join(
    [f"MRP-{1000+i},160.0,230.0,cm" for i in range(10_000)]
)

def direct_read():
    return pd.read_csv(io.StringIO(large_data))

def chunked_read():
    chunks = []
    for chunk in pd.read_csv(io.StringIO(large_data), chunksize=2000):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

t_direct = timeit.timeit(direct_read, number=5) / 5
t_chunked = timeit.timeit(chunked_read, number=5) / 5

print(f"10,000 Satır İçin Okuma Süreleri:")
print(f"- Tek Seferde Okuma (Direct) : {t_direct*1000:.2f} ms")
print(f"- Chunking ile Okuma (2000'er): {t_chunked*1000:.2f} ms")

10,000 Satır İçin Okuma Süreleri:
- Tek Seferde Okuma (Direct) : 6.60 ms
- Chunking ile Okuma (2000'er): 18.25 ms


## 7. Görselleştirme: Veri Kalite Hunisi (Data Quality Funnel)

Boru hattına giren ham kayıtların kaç tanesinin geçerli olduğunu, kaç tanesinin karantinaya ayrıldığını gösteren kalite dökümü:

In [4]:
# Veri Kalite İstatistikleri Tablosu
funnel_stats = pd.DataFrame([
    {"Asama": "1. Ham CSV Kayitlari", "Kayit_Sayisi": 10, "Yuzde": 100.0},
    {"Asama": "2. Ham JSON Kayitlari", "Kayit_Sayisi": 10, "Yuzde": 100.0},
    {"Asama": "3. Birlestirilmis Tekil Urun", "Kayit_Sayisi": 10, "Yuzde": 100.0},
    {"Asama": "4. Dogrulanmis & Gecerli (Pydantic)", "Kayit_Sayisi": 10, "Yuzde": 100.0},
    {"Asama": "5. Karantina (Hatali/Bozuk)", "Kayit_Sayisi": 0, "Yuzde": 0.0}
])

print("=== VERİ KALİTE HUNİSİ (DATA QUALITY FUNNEL) ===")
print(funnel_stats.to_string(index=False))

=== VERİ KALİTE HUNİSİ (DATA QUALITY FUNNEL) ===
                              Asama  Kayit_Sayisi  Yuzde
               1. Ham CSV Kayitlari            10  100.0
              2. Ham JSON Kayitlari            10  100.0
       3. Birlestirilmis Tekil Urun            10  100.0
4. Dogrulanmis & Gecerli (Pydantic)            10  100.0
        5. Karantina (Hatali/Bozuk)             0    0.0


## 8. Doğrulama ve Testler

Modül içindeki normalizer ve Pydantic model dökümünün doğruluğunu kontrol edelim.

In [5]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "day03" else CURRENT_DIR
DAY_DIR = PROJECT_ROOT / "day03"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from day03.mini_project.src.normalizer import DataNormalizer
norm = DataNormalizer()

# Test: inch -> cm dönüştürme doğruluğu
assert norm.convert_dimension_to_cm(100.0, "inch") == 254.0
# Test: 3 karakterlik hex'in 7 karaktere genişletilmesi
assert norm.normalize_hex_color("#abc") == "#AABBCC"
print("✅ Birim ve Renk dönüşüm testleri başarıyla doğrulandı!")

✅ Birim ve Renk dönüşüm testleri başarıyla doğrulandı!


## 9. Hata Durumları ve Karantina Mekanizması

Bozuk veya aykırı satırlar sistemde sessizce kaybolmaz; hata türüyle birlikte karantinaya yönlendirilir.

In [6]:
bad_sample = {
    "product_id": "MRP-9999",
    "product_name": "Hatali Boyutlu Hali",
    "collection": "Test",
    "composition": "wool",
    "raw_width": -50.0,  # Negatif genişlik!
    "raw_length": 200.0,
    "dimension_unit": "cm"
}

prod, err = norm.process_record(bad_sample)
assert prod is None
print("Karantinaya Alınan Hatalı Kayıt:")
print(json.dumps(err, indent=2, ensure_ascii=False))

Karantinaya Alınan Hatalı Kayıt:
{
  "product_id": "MRP-9999",
  "error_type": "ValidationError",
  "error_message": "1 validation error for CarpetDimensions\nwidth_cm\n  Input should be greater than or equal to 20 [type=greater_than_equal, input_value=-50.0, input_type=float]\n    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal",
  "raw_record": {
    "product_id": "MRP-9999",
    "product_name": "Hatali Boyutlu Hali",
    "collection": "Test",
    "composition": "wool",
    "raw_width": -50.0,
    "raw_length": 200.0,
    "dimension_unit": "cm"
  }
}


## 10. Mühendislik Çıkarımları & Sonraki Adım

### Çıkarımlar:
1. **Boru Hattı Dayanıklılığı (Resilience):** Tek bir hatalı satırın boru hattını durdurması engellenmiş, karantina mantığı ile veri akışı kesintisiz kılınmıştır.
2. **Çok Kaynaklı Bütünlük (Multi-Source Integrity):** Üretim tezgahı ile katalog verileri tek bir doğrulanmış şemada birleştirilmiştir.
3. **Tip Güvenliği:** Ham dizeler (string) güvenli enum ve Pydantic nesnelerine dönüştürülmüştür.

### Sonraki Adım (Day 04):
Day 04'te, bu temizlenmiş veri seti üzerinde **Great Expectations** ve veri profilleme (profiling) araçlarını kullanarak otomatik veri doğrulama raporlama hattı (EDA & Data Validation) kuracağız.